# Day 4: Tokens, Context Windows, and LLM Internals
Today we will learn how LLMs break down text and understand the memory limitations of models.

## What You'll Learn:
1. **Tokens:** How text is converted into integers under the hood.
2. **Counting Tokens:** Using the `tiktoken` library locally.
3. **Context Windows:** Understanding the limits of what a model can remember in a single session.
4. **Edge Cases:** Testing the limits of tokenizers (e.g. counting letters).

In [1]:
import sys
import subprocess

# 1. Automatically install tiktoken if it is missing
try:
    import tiktoken
    print("tiktoken is already installed.")
except ImportError:
    print("tiktoken not found. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken"])
    import tiktoken

# 2. Import standard libraries
import ollama

print("Day 4 workspace ready!")

tiktoken is already installed.
Day 4 workspace ready!


## What is a Token?
LLMs do not read text letter-by-letter or word-by-word like humans do. Instead, they break text down into chunks of characters called **tokens**. 

- A token can be a whole word (like `database`), part of a word (like `data`), or a single punctuation mark (like `!`).
- On average, **1 token is roughly 4 characters, or 0.75 words**.
- The model converts each token into a unique number (ID) before processing it.

To count tokens, we use a local library called **`tiktoken`** (created by OpenAI). Even though different models use slightly different tokenizers, `tiktoken` gives us a very close approximation for learning.

In [2]:
import tiktoken

# 1. Load the tokenizer for modern GPT models
tokenizer = tiktoken.get_encoding("cl100k_base")

# 2. Define a sentence to test
sample_text = "Hello, world! LLM Engineering is fun."

# 3. Convert the text into a list of Token IDs (numbers)
token_ids = tokenizer.encode(sample_text)

print("Original Text:")
print(sample_text)
print("-" * 50)

print("Token IDs (numbers sent to the model):")
print(token_ids)
print(f"Total tokens count: {len(token_ids)}")
print("-" * 50)

# 4. Let's see what word chunk each Token ID corresponds to!
print("Decoded token chunks:")
for token_id in token_ids:
    # Decode the single token ID back to text
    chunk_text = tokenizer.decode([token_id])
    print(f"Token ID {token_id} -> '{chunk_text}'")

Original Text:
Hello, world! LLM Engineering is fun.
--------------------------------------------------
Token IDs (numbers sent to the model):
[9906, 11, 1917, 0, 445, 11237, 17005, 374, 2523, 13]
Total tokens count: 10
--------------------------------------------------
Decoded token chunks:
Token ID 9906 -> 'Hello'
Token ID 11 -> ','
Token ID 1917 -> ' world'
Token ID 0 -> '!'
Token ID 445 -> ' L'
Token ID 11237 -> 'LM'
Token ID 17005 -> ' Engineering'
Token ID 374 -> ' is'
Token ID 2523 -> ' fun'
Token ID 13 -> '.'


## What is a Context Window?
The **Context Window** is the maximum amount of text (measured in tokens) that an LLM can read and write in a single conversation. Think of it as the model's short-term memory limit.

Every model has a strict limit:
- **`llama3.2:1b`:** 128,000 tokens (very large!).
- **`gemma3:270m`:** 32,000 tokens.
- **Older models (like GPT-3.5):** 4,096 tokens.

If you send a prompt that exceeds this limit, the model will either crash or forget the beginning of the conversation. As an AI engineer, you should check your token count before calling the API.

In [3]:
# A helper function to count tokens in a string
def get_token_count(text):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    return len(tokenizer.encode(text))

# Let's simulate a long document by repeating a sentence
long_article = "This is a paragraph about machine learning. " * 300

# 1. Count the tokens in our text
tokens_count = get_token_count(long_article)
print(f"Document length: {len(long_article)} characters")
print(f"Total tokens in document: {tokens_count} tokens")
print("-" * 50)

# 2. Check if it fits in a model's context window limit (e.g. let's set a safety threshold of 2,000 tokens)
safety_limit = 2000

if tokens_count <= safety_limit:
    print(f"Safe! {tokens_count} fits within the safety limit of {safety_limit} tokens.")
else:
    print(f"Too long! {tokens_count} exceeds the safety limit of {safety_limit} tokens.")
    print("Suggestion: Truncate or summarize the text before sending it to the model!")

Document length: 13200 characters
Total tokens in document: 2401 tokens
--------------------------------------------------
Too long! 2401 exceeds the safety limit of 2000 tokens.
Suggestion: Truncate or summarize the text before sending it to the model!


## Tokenizer Edge Cases: The "Strawberry" Problem
You might have seen memes online about advanced LLMs failing at simple questions, like: *"How many Rs are in the word 'strawberry'?"* 

Why does this happen?
Because of tokenization! The model doesn't see the letters `s-t-r-a-w-b-e-r-r-y`. It sees the word split into two tokens: `straw` and `berry`. 

Since it doesn't see the individual letters, it has to guess based on memory, which often leads to errors. Let's test this in Python!

In [4]:
prompt = "How many 'r's are in the word 'strawberry'?"

print(f"Question: '{prompt}'")
print("-" * 50)

# Call the model
response = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': prompt}]
)

# Print what the AI responds
print("Response:")
print(response.message.content)

Question: 'How many 'r's are in the word 'strawberry'?'
--------------------------------------------------
Response:
There are three 'r's in the word "strawberry".
